In [ ]:
"""
Project 01: Credit Risk Prediction
Step 03: Feature Engineering
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in globals() else (os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
ID_COL = "SK_ID_CURR"

In [ ]:
print("=" * 60)
print("STEP 03: Feature Engineering")
print("=" * 60)
start_time = time.time()

In [ ]:
# ═══════════════════════════════════════════════════════════
#  1. Bureau Features
# ═══════════════════════════════════════════════════════════
print("\n[1/3] Building Bureau Features ...")

In [ ]:
bureau = pd.read_csv(os.path.join(RAW, "bureau.csv"))
bureau_balance = pd.read_csv(os.path.join(RAW, "bureau_balance.csv"))
print(f"  bureau: {bureau.shape}, bureau_balance: {bureau_balance.shape}")

In [ ]:
# Bureau balance aggregates per SK_ID_BUREAU
print("  1a. Bureau balance aggregates ...")
status_order = {"C": 0, "X": 0, "0": 1, "1": 2, "2": 3, "3": 4, "4": 5, "5": 6}
bureau_balance["STATUS_ENC"] = bureau_balance["STATUS"].map(status_order).fillna(0)

In [ ]:
bal_agg = (
    bureau_balance.groupby("SK_ID_BUREAU")
    .agg(
        BUREAU_BAL_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
        BUREAU_BAL_MONTHS_MIN=("MONTHS_BALANCE", "min"),
        BUREAU_BAL_STATUS_MEAN=("STATUS_ENC", "mean"),
        BUREAU_BAL_STATUS_MAX=("STATUS_ENC", "max"),
        BUREAU_BAL_OVERDUE_COUNT=("STATUS_ENC", lambda x: (x > 1).sum()),
        BUREAU_BAL_DPD_RATE=(
            "STATUS_ENC",
            lambda x: (x > 1).sum() / len(x) if len(x) > 0 else 0,
        ),
    )
    .reset_index()
)

In [ ]:
bureau_enriched = bureau.merge(bal_agg, on="SK_ID_BUREAU", how="left")

In [ ]:
# Bureau main aggregates per SK_ID_CURR
print("  1b. Bureau main aggregates ...")
credit_counts = (
    bureau.groupby(ID_COL)["SK_ID_BUREAU"].count().rename("BUREAU_CREDIT_COUNT")
)

In [ ]:
amount_agg = bureau.groupby(ID_COL).agg(
    BUREAU_AMT_CREDIT_SUM_TOTAL=("AMT_CREDIT_SUM", "sum"),
    BUREAU_AMT_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_AMT_CREDIT_SUM_MAX=("AMT_CREDIT_SUM", "max"),
    BUREAU_AMT_CREDIT_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_AMT_CREDIT_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    BUREAU_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_DAYS_CREDIT_ENDDATE_MEAN=("DAYS_CREDIT_ENDDATE", "mean"),
)

In [ ]:
bureau_main = amount_agg.join(credit_counts, how="left").reset_index()

In [ ]:
# Active credit features
print("  1c. Active credit features ...")
active = bureau[bureau["CREDIT_ACTIVE"] == "Active"]
active_agg = (
    active.groupby(ID_COL)
    .agg(
        BUREAU_ACTIVE_COUNT=("SK_ID_BUREAU", "count"),
        BUREAU_ACTIVE_AMT_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
        BUREAU_ACTIVE_AMT_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_ACTIVE_AMT_CREDIT_MAX=("AMT_CREDIT_SUM", "max"),
        BUREAU_ACTIVE_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    )
    .reset_index()
)
active_agg["BUREAU_ACTIVE_DEBT_RATIO"] = np.where(
    active_agg["BUREAU_ACTIVE_AMT_CREDIT_SUM"] > 0,
    active_agg["BUREAU_ACTIVE_AMT_DEBT_SUM"]
    / active_agg["BUREAU_ACTIVE_AMT_CREDIT_SUM"],
    0,
)

In [ ]:
# DPD features
print("  1d. DPD features ...")
if "CREDIT_DAY_OVERDUE" in bureau.columns:
    dpd_agg = (
        bureau.groupby(ID_COL)
        .agg(
            BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
            BUREAU_CREDIT_DAY_OVERDUE_MEAN=("CREDIT_DAY_OVERDUE", "mean"),
            BUREAU_CREDIT_DAY_OVERDUE_SUM=("CREDIT_DAY_OVERDUE", "sum"),
            BUREAU_OVERDUE_AMOUNT_TOTAL=("AMT_CREDIT_SUM_OVERDUE", "sum"),
            BUREAU_OVERDUE_AMOUNT_MAX=("AMT_CREDIT_SUM_OVERDUE", "max"),
            BUREAU_LOANS_WITH_OVERDUE=(
                "AMT_CREDIT_SUM_OVERDUE",
                lambda x: (x > 0).sum(),
            ),
        )
        .reset_index()
    )
else:
    dpd_agg = pd.DataFrame({ID_COL: []})

In [ ]:
# Merge all bureau features
bureau_feats = bureau_main
for feats in [active_agg, dpd_agg]:
    bureau_feats = bureau_feats.merge(feats, on=ID_COL, how="left")
bureau_feats = bureau_feats.fillna(0)

In [ ]:
del bureau, bureau_balance, bal_agg, bureau_enriched
print(f"  Bureau features shape: {bureau_feats.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  2. Application Risk Features
# ═══════════════════════════════════════════════════════════
print("\n[2/3] Building Application Risk Features ...")

In [ ]:
app_train = pd.read_csv(os.path.join(RAW, "application_train.csv"))

In [ ]:
# Handle anomaly
app_train["DAYS_EMPLOYED"] = app_train["DAYS_EMPLOYED"].replace(365243, np.nan)

In [ ]:
# Income features
app_train["RISK_CREDIT_TO_INCOME_RATIO"] = np.where(
    app_train["AMT_INCOME_TOTAL"] > 0,
    app_train["AMT_CREDIT"] / app_train["AMT_INCOME_TOTAL"],
    np.nan,
)
app_train["RISK_ANNUITY_TO_INCOME_RATIO"] = np.where(
    app_train["AMT_INCOME_TOTAL"] > 0,
    app_train["AMT_ANNUITY"] / app_train["AMT_INCOME_TOTAL"],
    np.nan,
)
if "AMT_GOODS_PRICE" in app_train.columns:
    app_train["RISK_CREDIT_TO_GOODS_RATIO"] = np.where(
        app_train["AMT_GOODS_PRICE"] > 0,
        app_train["AMT_CREDIT"] / app_train["AMT_GOODS_PRICE"],
        np.nan,
    )
app_train["RISK_LOAN_TERM_MONTHS"] = np.where(
    app_train["AMT_ANNUITY"] > 0,
    app_train["AMT_CREDIT"] / app_train["AMT_ANNUITY"],
    np.nan,
)
if "CNT_FAM_MEMBERS" in app_train.columns:
    app_train["RISK_INCOME_PER_PERSON"] = np.where(
        app_train["CNT_FAM_MEMBERS"] > 0,
        app_train["AMT_INCOME_TOTAL"] / app_train["CNT_FAM_MEMBERS"],
        app_train["AMT_INCOME_TOTAL"],
    )

In [ ]:
# Employment features
app_train["RISK_EMPLOYMENT_YEARS"] = np.where(
    app_train["DAYS_EMPLOYED"].isnull() | (app_train["DAYS_EMPLOYED"] >= 0),
    np.nan,
    np.abs(app_train["DAYS_EMPLOYED"]) / 365,
)
app_train["RISK_IS_UNEMPLOYED"] = (
    app_train["DAYS_EMPLOYED"].isnull() | (app_train["DAYS_EMPLOYED"] >= 0)
).astype(int)
app_train["RISK_AGE_YEARS"] = np.abs(app_train["DAYS_BIRTH"]) / 365
app_train["RISK_EMPLOYED_TO_AGE_RATIO"] = np.where(
    (app_train["RISK_AGE_YEARS"] > 0) & (~app_train["RISK_EMPLOYMENT_YEARS"].isnull()),
    app_train["RISK_EMPLOYMENT_YEARS"] / app_train["RISK_AGE_YEARS"],
    np.nan,
)

In [ ]:
# External score features
ext_cols = [
    c
    for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    if c in app_train.columns
]
if ext_cols:
    app_train["RISK_EXT_SOURCE_MEAN"] = app_train[ext_cols].mean(axis=1)
    if len(ext_cols) > 1:
        app_train["RISK_EXT_SOURCE_STD"] = app_train[ext_cols].std(axis=1)
    app_train["RISK_EXT_SOURCE_MIN"] = app_train[ext_cols].min(axis=1)
    app_train["RISK_EXT_SOURCE_MAX"] = app_train[ext_cols].max(axis=1)
    app_train["RISK_EXT_SOURCE_PRODUCT"] = app_train[ext_cols].prod(axis=1, skipna=True)
    app_train["RISK_EXT_SOURCE_MISSING_COUNT"] = (
        app_train[ext_cols].isnull().sum(axis=1)
    )

In [ ]:
# Document features
doc_cols = [c for c in app_train.columns if c.startswith("FLAG_DOCUMENT_")]
if doc_cols:
    app_train["RISK_DOCS_SUBMITTED_COUNT"] = app_train[doc_cols].sum(axis=1)
    app_train["RISK_DOCS_SUBMISSION_RATE"] = app_train[doc_cols].mean(axis=1)

In [ ]:
# Demographic features
app_train["RISK_AGE_GROUP"] = pd.cut(
    app_train["RISK_AGE_YEARS"], bins=[0, 25, 35, 45, 55, 100], labels=[0, 1, 2, 3, 4]
).astype(float)
if "CODE_GENDER" in app_train.columns:
    app_train["RISK_IS_FEMALE"] = (app_train["CODE_GENDER"] == "F").astype(int)
if "FLAG_OWN_CAR" in app_train.columns:
    app_train["RISK_OWNS_CAR"] = (app_train["FLAG_OWN_CAR"] == "Y").astype(int)
if "FLAG_OWN_REALTY" in app_train.columns:
    app_train["RISK_OWNS_REALTY"] = (app_train["FLAG_OWN_REALTY"] == "Y").astype(int)

In [ ]:
risk_cols = [c for c in app_train.columns if c.startswith("RISK_")]
print(f"  Risk features: {len(risk_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  3. Behaviour Features
# ═══════════════════════════════════════════════════════════
print("\n[3/3] Building Behaviour Features ...")

In [ ]:
# Previous application features
print("  3a. Previous application features ...")
prev = pd.read_csv(os.path.join(RAW, "previous_application.csv"))

In [ ]:
prev_count = prev.groupby(ID_COL)["SK_ID_PREV"].count().rename("BEH_PREV_APP_COUNT")
if "NAME_CONTRACT_STATUS" in prev.columns:
    approved = (
        prev[prev["NAME_CONTRACT_STATUS"] == "Approved"]
        .groupby(ID_COL)["SK_ID_PREV"]
        .count()
        .rename("BEH_PREV_APPROVED_COUNT")
    )
    refused = (
        prev[prev["NAME_CONTRACT_STATUS"] == "Refused"]
        .groupby(ID_COL)["SK_ID_PREV"]
        .count()
        .rename("BEH_PREV_REFUSED_COUNT")
    )
else:
    approved = pd.Series(dtype=float, name="BEH_PREV_APPROVED_COUNT")
    refused = pd.Series(dtype=float, name="BEH_PREV_REFUSED_COUNT")

In [ ]:
prev_agg = (
    prev.groupby(ID_COL)
    .agg(
        BEH_PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
        BEH_PREV_AMT_CREDIT_MAX=("AMT_CREDIT", "max"),
        BEH_PREV_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
        BEH_PREV_AMT_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
        BEH_PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    )
    .reset_index()
)

In [ ]:
prev_feats = (
    prev_agg.set_index(ID_COL)
    .join(prev_count, how="left")
    .join(approved, how="left")
    .join(refused, how="left")
    .reset_index()
)
prev_feats["BEH_PREV_APPROVAL_RATE"] = np.where(
    prev_feats["BEH_PREV_APP_COUNT"] > 0,
    prev_feats["BEH_PREV_APPROVED_COUNT"].fillna(0) / prev_feats["BEH_PREV_APP_COUNT"],
    np.nan,
)
prev_feats["BEH_PREV_REFUSAL_RATE"] = np.where(
    prev_feats["BEH_PREV_APP_COUNT"] > 0,
    prev_feats["BEH_PREV_REFUSED_COUNT"].fillna(0) / prev_feats["BEH_PREV_APP_COUNT"],
    np.nan,
)
del prev

In [ ]:
# Installment features
print("  3b. Installment payment features ...")
inst = pd.read_csv(os.path.join(RAW, "installments_payments.csv"))
inst["INSTALMENT_DELAY_DAYS"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
inst["INSTALMENT_UNDERPAYMENT"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]
inst["INSTALMENT_UNDERPAYMENT_RATIO"] = np.where(
    inst["AMT_INSTALMENT"] > 0,
    inst["INSTALMENT_UNDERPAYMENT"] / inst["AMT_INSTALMENT"],
    0,
)
inst["INSTALMENT_PAID_LATE"] = (inst["INSTALMENT_DELAY_DAYS"] > 0).astype(int)
inst["INSTALMENT_UNDERPAID"] = (inst["INSTALMENT_UNDERPAYMENT"] > 0).astype(int)

In [ ]:
inst_feats = (
    inst.groupby(ID_COL)
    .agg(
        BEH_INST_COUNT=("AMT_INSTALMENT", "count"),
        BEH_INST_DELAY_MEAN=("INSTALMENT_DELAY_DAYS", "mean"),
        BEH_INST_DELAY_MAX=("INSTALMENT_DELAY_DAYS", "max"),
        BEH_INST_LATE_COUNT=("INSTALMENT_PAID_LATE", "sum"),
        BEH_INST_LATE_RATE=("INSTALMENT_PAID_LATE", "mean"),
        BEH_INST_UNDERPAYMENT_MEAN=("INSTALMENT_UNDERPAYMENT", "mean"),
        BEH_INST_UNDERPAYMENT_RATIO_MEAN=("INSTALMENT_UNDERPAYMENT_RATIO", "mean"),
        BEH_INST_UNDERPAID_RATE=("INSTALMENT_UNDERPAID", "mean"),
        BEH_INST_AMT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    )
    .reset_index()
)
del inst

In [ ]:
# POS cash features
print("  3c. POS cash balance features ...")
pos = pd.read_csv(os.path.join(RAW, "POS_CASH_balance.csv"))
pos = pos[pos["MONTHS_BALANCE"] >= -12]

In [ ]:
pos_feats = (
    pos.groupby(ID_COL)
    .agg(
        BEH_POS_COUNT=("SK_ID_PREV", "count"),
        BEH_POS_MONTHS_BALANCE_MEAN=("MONTHS_BALANCE", "mean"),
        BEH_POS_CNT_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
        BEH_POS_CNT_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    )
    .reset_index()
)

In [ ]:
if "SK_DPD" in pos.columns:
    pos_dpd = (
        pos.groupby(ID_COL)
        .agg(
            BEH_POS_DPD_MEAN=("SK_DPD", "mean"),
            BEH_POS_DPD_MAX=("SK_DPD", "max"),
            BEH_POS_DPD_GT0_RATE=(
                "SK_DPD",
                lambda x: (x > 0).mean() if len(x) > 0 else 0,
            ),
        )
        .reset_index()
    )
    pos_feats = pos_feats.merge(pos_dpd, on=ID_COL, how="left")
del pos

In [ ]:
# Credit card features
print("  3d. Credit card balance features ...")
cc = pd.read_csv(os.path.join(RAW, "credit_card_balance.csv"))
cc = cc[cc["MONTHS_BALANCE"] >= -12]

In [ ]:
cc_agg_dict = {"BEH_CC_COUNT": ("SK_ID_PREV", "count")}
if "AMT_BALANCE" in cc.columns:
    cc_agg_dict["BEH_CC_AMT_BALANCE_MEAN"] = ("AMT_BALANCE", "mean")
    cc_agg_dict["BEH_CC_AMT_BALANCE_MAX"] = ("AMT_BALANCE", "max")
if "AMT_CREDIT_LIMIT_ACTUAL" in cc.columns:
    cc_agg_dict["BEH_CC_CREDIT_LIMIT_MEAN"] = ("AMT_CREDIT_LIMIT_ACTUAL", "mean")
if "AMT_DRAWINGS_CURRENT" in cc.columns:
    cc_agg_dict["BEH_CC_DRAWINGS_MEAN"] = ("AMT_DRAWINGS_CURRENT", "mean")
if "AMT_PAYMENT_CURRENT" in cc.columns:
    cc_agg_dict["BEH_CC_PAYMENT_MEAN"] = ("AMT_PAYMENT_CURRENT", "mean")

In [ ]:
cc_feats = cc.groupby(ID_COL).agg(**cc_agg_dict).reset_index()
if (
    "BEH_CC_AMT_BALANCE_MEAN" in cc_feats.columns
    and "BEH_CC_CREDIT_LIMIT_MEAN" in cc_feats.columns
):
    cc_feats["BEH_CC_UTILISATION_RATIO"] = np.where(
        cc_feats["BEH_CC_CREDIT_LIMIT_MEAN"] > 0,
        cc_feats["BEH_CC_AMT_BALANCE_MEAN"] / cc_feats["BEH_CC_CREDIT_LIMIT_MEAN"],
        np.nan,
    )

In [ ]:
if "SK_DPD" in cc.columns:
    cc_dpd = (
        cc.groupby(ID_COL)
        .agg(
            BEH_CC_DPD_MEAN=("SK_DPD", "mean"),
            BEH_CC_DPD_MAX=("SK_DPD", "max"),
            BEH_CC_DPD_GT0_RATE=(
                "SK_DPD",
                lambda x: (x > 0).mean() if len(x) > 0 else 0,
            ),
        )
        .reset_index()
    )
    cc_feats = cc_feats.merge(cc_dpd, on=ID_COL, how="left")
del cc

In [ ]:
# Merge behaviour features
beh_feats = prev_feats
for feats in tqdm([inst_feats, pos_feats, cc_feats], desc="  Merging behaviour"):
    beh_feats = beh_feats.merge(feats, on=ID_COL, how="left")
beh_feats = beh_feats.fillna(0)

In [ ]:
print(f"  Behaviour features shape: {beh_feats.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Final Merge
# ═══════════════════════════════════════════════════════════
print("\nMerging all features ...")
final = app_train.merge(bureau_feats, on=ID_COL, how="left")
final = final.merge(beh_feats, on=ID_COL, how="left")
final = final.fillna(0)

In [ ]:
print(f"Final shape: {final.shape[0]:,} rows × {final.shape[1]} columns")

In [ ]:
# Save
train_path = os.path.join(PROCESSED, "train_features.parquet")
final.to_parquet(train_path, index=False)
print(f"Saved: {train_path}")

In [ ]:
csv_path = os.path.join(PROCESSED, "train_features.csv")
final.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

In [ ]:
# Feature count chart
fig, ax = plt.subplots(figsize=(10, 5))
categories = {
    "Bureau": len([c for c in final.columns if c.startswith("BUREAU_")]),
    "Risk": len([c for c in final.columns if c.startswith("RISK_")]),
    "Behaviour": len([c for c in final.columns if c.startswith("BEH_")]),
    "Original": len(
        [c for c in final.columns if not c.startswith(("BUREAU_", "RISK_", "BEH_"))]
    ),
}
ax.bar(
    categories.keys(),
    categories.values(),
    color=["#6C63FF", "#FF6584", "#43E97B", "#FFB347"],
    alpha=0.85,
)
for i, (k, v) in enumerate(categories.items()):
    ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
ax.set_title("Feature Categories After Engineering", fontweight="bold")
ax.set_ylabel("Feature Count")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "feature_categories.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"Saved chart: feature_categories.png")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 03 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")